# 📊 Analisis Data Eksploratif (EDA) — Aplikasi Jenius

**Sumber Data:** Google Play Store  
**App ID:** `com.btpn.dc`  
**Nama Aplikasi:** Jenius — BTPN Digital Banking  

---

## 📋 Rangkaian Analisis
1. Instalasi & Import Library
2. Load Dataset
3. Transformasi Format Data Waktu
4. Distribusi Skor Ulasan
5. Tren Frekuensi Ulasan Tahunan
6. Tren Rata-rata Skor Tahunan
7. Distribusi Panjang Ulasan
8. Statistik Panjang Ulasan per Skor
9. Frekuensi Kata (Word Frequency)
10. Responsivitas Pengembang
11. Kecepatan Respons Pengembang
12. **[Tambahan]** Distribusi Ulasan per Bulan
13. **[Tambahan]** Analisis Versi Aplikasi
14. **[Tambahan]** Proporsi Ulasan Mendapat Thumbs Up

---
## 1. 📦 Instalasi & Import Library

In [ ]:
!pip install matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

# Konfigurasi tampilan
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

available_styles = matplotlib.style.available
if 'seaborn-v0_8' in available_styles:
    plt.style.use('seaborn-v0_8')
elif 'seaborn' in available_styles:
    plt.style.use('seaborn')
else:
    plt.style.use('ggplot')

pd.set_option('display.max_colwidth', 200)
print('✅ Library berhasil diimport!')

---
## 2. 📂 Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load dataset ulasan Jenius
df_jenius = pd.read_csv('/content/drive/My Drive/PBA/rawdata_jenius_id.csv', low_memory=False)
df_jenius.head()

In [ ]:
# Ringkasan informasi dataset
df_jenius.info()

---
## 3. 🕐 Transformasi Format Data Waktu

Kolom `at` yang semula bertipe string dikonversi ke format `datetime` agar analisis berbasis waktu dapat dilakukan secara kronologis. Validasi tipe data dilakukan setelah proses konversi untuk memastikan struktur kolom telah sesuai.

In [ ]:
df_jenius['at'] = pd.to_datetime(df_jenius['at'])
df_jenius.info()

---
## 4. ⭐ Distribusi Skor Ulasan

Perhitungan proporsi jumlah ulasan pada setiap tingkatan rating (1–5 bintang) dilakukan untuk mendeteksi potensi ketidakseimbangan data (*data imbalance*) yang dapat memengaruhi hasil pemodelan sentimen.

In [ ]:
df_jenius['score'].value_counts()

In [ ]:
df_count = df_jenius['score'].value_counts().sort_index()

score_colors = {
    1: '#e74c3c',
    2: '#e67e22',
    3: '#f1c40f',
    4: '#bffc6b',
    5: '#0aa800',
}
colors = [score_colors[s] for s in df_count.index]

plt.figure(figsize=(10, 6))
bars = plt.bar(df_count.index, df_count.values, color=colors, edgecolor='black', linewidth=1)

plt.title('Distribusi Skor Ulasan Jenius', fontsize=20, fontweight='bold', pad=25)
plt.xlabel('Skor', fontsize=14)
plt.ylabel('Jumlah Ulasan', fontsize=14)

max_val = max(df_count.values)
plt.ylim(top=max_val * 1.15)
plt.grid(axis='y', linestyle='--', alpha=0.6)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + max_val * 0.03,
        f"{height:,}",
        ha='center', va='bottom', fontsize=12
    )

plt.tight_layout()
plt.show()

---
## 5. 📈 Tren Frekuensi Ulasan Tahunan

Analisis volume ulasan yang masuk per tahun dilakukan untuk memetakan dinamika pertumbuhan pengguna aplikasi Jenius dari waktu ke waktu. Observasi ini penting untuk memastikan dataset mencakup rentang waktu yang representatif.

In [ ]:
df_yearly = df_jenius.groupby(df_jenius['at'].dt.year).size()

plt.figure(figsize=(10, 5))
plt.plot(df_yearly.index, df_yearly.values, marker='o', linewidth=2, color='steelblue')
plt.fill_between(df_yearly.index, df_yearly.values, alpha=0.15, color='steelblue')

plt.title('Jumlah Ulasan per Tahun — Jenius', fontsize=16, fontweight='bold')
plt.xlabel('Tahun', fontsize=13)
plt.ylabel('Jumlah Ulasan', fontsize=13)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. 📉 Tren Rata-rata Skor Tahunan

Evaluasi fluktuasi tingkat kepuasan pengguna dilakukan dengan menghitung rata-rata skor ulasan per tahun untuk mengamati apakah kualitas pengalaman pengguna cenderung meningkat, stabil, atau menurun secara kronologis.

In [ ]:
df_yearly_score = df_jenius.groupby(df_jenius['at'].dt.year)['score'].mean()

plt.figure(figsize=(10, 5))
plt.plot(df_yearly_score.index, df_yearly_score.values, marker='o', linewidth=2, color='coral')
plt.fill_between(df_yearly_score.index, df_yearly_score.values, alpha=0.15, color='coral')

plt.title('Rata-rata Skor Ulasan per Tahun — Jenius', fontsize=16, fontweight='bold')
plt.xlabel('Tahun', fontsize=13)
plt.ylabel('Rata-rata Skor', fontsize=13)
plt.ylim(1, 5.5)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. 📏 Distribusi Panjang Ulasan

Sebaran jumlah karakter pada setiap ulasan dianalisis melalui histogram untuk memahami kecenderungan pengguna dalam menyampaikan umpan balik, serta mendeteksi keberadaan pencilan (*outliers*) pada dataset.

In [ ]:
df_jenius['text_length'] = df_jenius['content'].astype(str).apply(len)

plt.figure(figsize=(10, 5))
plt.hist(df_jenius['text_length'], bins=50, color='steelblue', edgecolor='white')
plt.title('Distribusi Panjang Ulasan — Jenius', fontsize=16, fontweight='bold')
plt.xlabel('Jumlah Karakter', fontsize=13)
plt.ylabel('Frekuensi', fontsize=13)
plt.xlim(0, 600)
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

---
## 8. 📋 Statistik Panjang Ulasan per Skor

Hubungan antara tingkat kepuasan (skor rating) dan panjang teks ulasan ditelaah melalui perhitungan statistik deskriptif. Analisis ini memberikan gambaran mengenai pola perilaku pengguna dalam memberikan deskripsi ulasan pada setiap kategori skor.

In [ ]:
df_jenius['text_length'] = df_jenius['content'].astype(str).apply(len)
df_jenius.groupby('score')['text_length'].agg(['count', 'mean', 'median', 'max'])

In [ ]:
# Visualisasi rata-rata panjang ulasan per skor
avg_len = df_jenius.groupby('score')['text_length'].mean()

plt.figure(figsize=(9, 5))
bars = plt.bar(avg_len.index, avg_len.values,
               color=[score_colors[s] for s in avg_len.index],
               edgecolor='black', linewidth=1)

plt.title('Rata-rata Panjang Ulasan per Skor — Jenius', fontsize=15, fontweight='bold')
plt.xlabel('Skor', fontsize=13)
plt.ylabel('Rata-rata Jumlah Karakter', fontsize=13)
plt.grid(axis='y', linestyle='--', alpha=0.5)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 1,
             f"{height:.0f}", ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

---
## 9. 🔤 Analisis Frekuensi Kata

Kata kunci dominan yang paling sering muncul dalam korpus ulasan diidentifikasi menggunakan ekspresi reguler dengan filter panjang minimal tiga karakter untuk menyaring partikel kata yang tidak bermakna. Hasil 20 kata teratas divisualisasikan dalam diagram batang horizontal.

In [ ]:
all_text = " ".join(df_jenius['content'].dropna()).lower()
words = re.findall(r'\b[a-zA-Z]{3,}\b', all_text)

word_counts = Counter(words).most_common(20)
labels, values = zip(*word_counts)

plt.figure(figsize=(10, 6))
plt.barh(labels, values, color='steelblue', edgecolor='white')
plt.title('20 Kata Paling Sering Muncul — Jenius', fontsize=15, fontweight='bold')
plt.xlabel('Frekuensi', fontsize=13)
plt.gca().invert_yaxis()
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---
## 10. 💬 Analisis Responsivitas Pengembang

Tingkat interaksi pengembang terhadap umpan balik pengguna diukur dengan mengidentifikasi keberadaan konten balasan pada atribut `replyContent`. Hasil dikategorikan dan divisualisasikan dalam diagram lingkaran untuk menunjukkan proporsi responsivitas secara keseluruhan.

In [ ]:
df_jenius['is_replied'] = df_jenius['replyContent'].notnull()
reply_counts = df_jenius['is_replied'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(
    reply_counts,
    labels=["Tidak Dibalas", "Dibalas"],
    autopct="%1.1f%%",
    colors=['#e74c3c', '#2ecc71'],
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
plt.title('Persentase Ulasan yang Dibalas Pengembang — Jenius', fontsize=13, fontweight='bold')
plt.show()

total = reply_counts.sum()
print(f'Dibalas     : {reply_counts[True]:,} ({reply_counts[True]/total*100:.1f}%)')
print(f'Tidak dibalas: {reply_counts[False]:,} ({reply_counts[False]/total*100:.1f}%)')

---
## 11. ⚡ Analisis Kecepatan Respons Pengembang

Efisiensi pengembang dalam menanggapi ulasan diukur melalui perhitungan durasi waktu respons dalam satuan hari, kemudian dikategorikan ke dalam beberapa interval waktu untuk mengevaluasi tingkat kesigapan penanganan umpan balik.

In [ ]:
df_reply = df_jenius[df_jenius['replyContent'].notna()].copy()
df_reply['at'] = pd.to_datetime(df_reply['at'])
df_reply['repliedAt'] = pd.to_datetime(df_reply['repliedAt'])
df_reply['response_days'] = (df_reply['repliedAt'] - df_reply['at']).dt.days

df_reply['response_category'] = pd.cut(
    df_reply['response_days'],
    bins=[0, 1, 3, 7, 14, 30, 365],
    labels=["<1 hari", "1–3 hari", "3–7 hari", "7–14 hari", "14–30 hari", ">30 hari"]
)

resp_counts = df_reply['response_category'].value_counts().sort_index()
print(resp_counts)

# Visualisasi
plt.figure(figsize=(10, 5))
plt.bar(resp_counts.index, resp_counts.values, color='#3498db', edgecolor='black')
plt.title('Distribusi Kecepatan Respons Pengembang — Jenius', fontsize=15, fontweight='bold')
plt.xlabel('Kategori Waktu Respons', fontsize=13)
plt.ylabel('Jumlah Ulasan', fontsize=13)
plt.grid(axis='y', linestyle='--', alpha=0.5)
for i, v in enumerate(resp_counts.values):
    plt.text(i, v + 5, f"{v:,}", ha='center', fontsize=11)
plt.tight_layout()
plt.show()

---
## 12. 📅 [Tambahan] Distribusi Ulasan per Bulan

Analisis granularitas bulanan dilakukan untuk mengamati fluktuasi volume ulasan secara lebih rinci. Pola musiman atau lonjakan ulasan pada bulan tertentu dapat mengindikasikan adanya kejadian signifikan seperti pembaruan fitur, gangguan sistem (*downtime*), atau perubahan kebijakan layanan yang berdampak pada respons pengguna.

In [ ]:
# Ambil data 2 tahun terakhir agar visualisasi lebih fokus
df_recent = df_jenius[df_jenius['at'].dt.year >= df_jenius['at'].dt.year.max() - 1]
df_monthly = df_recent.groupby(df_recent['at'].dt.to_period('M')).size()

plt.figure(figsize=(14, 5))
plt.plot(df_monthly.index.astype(str), df_monthly.values,
         marker='o', linewidth=2, color='purple')
plt.fill_between(df_monthly.index.astype(str), df_monthly.values, alpha=0.15, color='purple')

plt.title('Tren Volume Ulasan per Bulan (2 Tahun Terakhir) — Jenius',
          fontsize=14, fontweight='bold')
plt.xlabel('Bulan', fontsize=12)
plt.ylabel('Jumlah Ulasan', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 13. 📱 [Tambahan] Analisis Versi Aplikasi

Distribusi ulasan berdasarkan versi aplikasi dianalisis untuk mengidentifikasi versi mana yang paling banyak mendapatkan umpan balik dari pengguna. Selain itu, perbandingan rata-rata skor per versi dapat mengungkap apakah pembaruan tertentu berdampak positif atau negatif terhadap pengalaman pengguna.

In [ ]:
# Top 10 versi dengan ulasan terbanyak
top_versions = df_jenius['reviewCreatedVersion'].value_counts().head(10)

plt.figure(figsize=(10, 5))
plt.barh(top_versions.index.astype(str), top_versions.values,
         color='#27ae60', edgecolor='white')
plt.title('Top 10 Versi Aplikasi dengan Ulasan Terbanyak — Jenius',
          fontsize=14, fontweight='bold')
plt.xlabel('Jumlah Ulasan', fontsize=12)
plt.ylabel('Versi Aplikasi', fontsize=12)
plt.gca().invert_yaxis()
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Rata-rata skor per versi (top 10 versi terbanyak)
top_ver_list = top_versions.index.tolist()
avg_score_ver = (
    df_jenius[df_jenius['reviewCreatedVersion'].isin(top_ver_list)]
    .groupby('reviewCreatedVersion')['score']
    .mean()
    .reindex(top_ver_list)
)

plt.figure(figsize=(10, 5))
bars = plt.barh(avg_score_ver.index.astype(str), avg_score_ver.values,
                color='#e67e22', edgecolor='white')
plt.title('Rata-rata Skor per Versi Aplikasi (Top 10) — Jenius',
          fontsize=14, fontweight='bold')
plt.xlabel('Rata-rata Skor', fontsize=12)
plt.ylabel('Versi Aplikasi', fontsize=12)
plt.xlim(0, 5.5)
plt.axvline(x=3, color='red', linestyle='--', alpha=0.5, label='Skor Netral (3)')
plt.legend()
plt.gca().invert_yaxis()
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---
## 14. 👍 [Tambahan] Proporsi Ulasan dengan Thumbs Up

Analisis keterlibatan komunitas (*community engagement*) dilakukan dengan menelaah distribusi jumlah thumbs up yang diterima setiap ulasan. Ulasan dengan thumbs up tinggi mengindikasikan bahwa konten ulasan tersebut dianggap relevan atau mewakilkan pendapat banyak pengguna lainnya, sehingga bobot informasinya lebih signifikan dalam memahami sentimen komunitas secara keseluruhan.

In [ ]:
# Kategorikan thumbs up
def kategori_thumbs(n):
    if n == 0:
        return '0 (Tidak Ada)'
    elif n <= 5:
        return '1–5'
    elif n <= 20:
        return '6–20'
    elif n <= 100:
        return '21–100'
    else:
        return '>100'

df_jenius['thumbs_category'] = df_jenius['thumbsUpCount'].apply(kategori_thumbs)
order = ['0 (Tidak Ada)', '1–5', '6–20', '21–100', '>100']
thumbs_dist = df_jenius['thumbs_category'].value_counts().reindex(order)

plt.figure(figsize=(10, 5))
bars = plt.bar(thumbs_dist.index, thumbs_dist.values,
               color=['#bdc3c7','#3498db','#2980b9','#1a6fa8','#0d4f7c'],
               edgecolor='white')
plt.title('Distribusi Jumlah Thumbs Up pada Ulasan — Jenius',
          fontsize=14, fontweight='bold')
plt.xlabel('Kategori Thumbs Up', fontsize=12)
plt.ylabel('Jumlah Ulasan', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.5)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 50,
             f"{height:,}", ha='center', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Top 5 ulasan dengan thumbs up terbanyak
print('=== TOP 5 ULASAN PALING BANYAK THUMBS UP ===')
top_thumbs = df_jenius.nlargest(5, 'thumbsUpCount')[['content', 'score', 'thumbsUpCount', 'at']]
display(top_thumbs)